<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/GammaFest_IPB/4_Model_XGBPoisson_AW_MAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
"""
AW-MAE — Augmented Weighted Mean Absolute Error
================================================
Implementasi lengkap metrik evaluasi kompetisi + strategi optimasi.

Komponen:
  1. Base MAE per pertandingan
  2. Penalty: Exact Score, Outcome (W/D/L), Goal Difference
  3. Outcome Multiplier (1.0 jika benar, 1.5 jika salah)
  4. Non-linear scaling (pangkat)
  5. Tournament weighting
"""

import numpy as np
import pandas as pd

In [21]:

# ═══════════════════════════════════════════════════════════════
# TOURNAMENT WEIGHTS
# ═══════════════════════════════════════════════════════════════
TOURNAMENT_WEIGHTS = {
    # FIFA
    "FIFA World Cup":                        2.00,
    "FIFA World Cup qualification":          1.60,
    "FIFA World Cup qualification, CONMEBOL":1.60,
    "FIFA World Cup qualification, UEFA":    1.60,
    "FIFA World Cup qualification, AFC":     1.60,
    "FIFA World Cup qualification, CAF":     1.60,
    "FIFA World Cup qualification, CONCACAF":1.60,
    "FIFA World Cup qualification, OFC":     1.60,
    "Confederations Cup":                    1.75,
    # Continental championships
    "UEFA Euro":                             1.90,
    "UEFA Euro qualification":               1.55,
    "Copa América":                          1.85,
    "Africa Cup of Nations":                 1.80,
    "AFC Asian Cup":                         1.80,
    "AFC Championship":                      1.80,
    "CONCACAF Championship":                 1.75,
    "CONCACAF Gold Cup":                     1.70,
    "OFC Nations Cup":                       1.65,
    # UEFA Nations League
    "UEFA Nations League":                   1.50,
    "UEFA Nations League A":                 1.55,
    "UEFA Nations League B":                 1.50,
    "UEFA Nations League C":                 1.45,
    "UEFA Nations League D":                 1.40,
    # Friendlies
    "Friendly":                              0.96,
    "Friendlies":                            0.96,
    # Default untuk turnamen tidak dikenal
    "__default__":                           1.20,
}


def get_tournament_weight(tournament_name: str) -> float:
    if pd.isna(tournament_name):
        return TOURNAMENT_WEIGHTS["__default__"]
    # Exact match dulu
    if tournament_name in TOURNAMENT_WEIGHTS:
        return TOURNAMENT_WEIGHTS[tournament_name]
    # Partial match
    t_lower = str(tournament_name).lower()
    if "world cup" in t_lower:
        return 1.60 if "qual" in t_lower else 2.00
    if "friendly" in t_lower:
        return 0.96
    if "euro" in t_lower:
        return 1.55 if "qual" in t_lower else 1.90
    if "copa" in t_lower or "america" in t_lower:
        return 1.85
    if "africa" in t_lower or "afcon" in t_lower:
        return 1.80
    if "asian" in t_lower or "afc" in t_lower:
        return 1.80
    if "concacaf" in t_lower or "gold cup" in t_lower:
        return 1.70
    if "nations league" in t_lower:
        return 1.50
    if "olympic" in t_lower:
        return 1.65
    return TOURNAMENT_WEIGHTS["__default__"]



In [22]:


# ═══════════════════════════════════════════════════════════════
# PENALTY CONSTANTS
# ═══════════════════════════════════════════════════════════════
PENALTY_EXACT_SCORE    = 0.30
PENALTY_OUTCOME        = 0.25
PENALTY_GOAL_DIFF      = 0.15
OUTCOME_MULTIPLIER_BAD = 1.50
OUTCOME_MULTIPLIER_OK  = 1.00
NONLINEAR_POWER        = 1.50   # pangkat non-linear (asumsi — sesuaikan jika tahu nilai pasti)


In [23]:


# ═══════════════════════════════════════════════════════════════
# HELPER: OUTCOME (W/D/L)
# ═══════════════════════════════════════════════════════════════
def get_outcome(team_goals: int, opp_goals: int) -> str:
    if team_goals > opp_goals:  return "W"
    if team_goals < opp_goals:  return "L"
    return "D"



In [24]:

# ═══════════════════════════════════════════════════════════════
# CORE: HITUNG LOSS SATU PERTANDINGAN
# ═══════════════════════════════════════════════════════════════
def compute_loss(
    pred_team: int, pred_opp: int,
    true_team: int, true_opp: int,
) -> float:
    # 1. Base MAE
    base_mae = (abs(pred_team - true_team) + abs(pred_opp - true_opp)) / 2.0

    # 2. Penalties
    exact_ok   = int(pred_team == true_team and pred_opp == true_opp)
    outcome_ok = int(get_outcome(pred_team, pred_opp) == get_outcome(true_team, true_opp))
    gd_ok      = int((pred_team - pred_opp) == (true_team - true_opp))

    penalty = (
        PENALTY_EXACT_SCORE * (1 - exact_ok) +
        PENALTY_OUTCOME     * (1 - outcome_ok) +
        PENALTY_GOAL_DIFF   * (1 - gd_ok)
    )

    # 3. Outcome multiplier
    multiplier = OUTCOME_MULTIPLIER_OK if outcome_ok else OUTCOME_MULTIPLIER_BAD

    # 4. Combined + non-linear scaling
    combined = (base_mae + penalty) * multiplier
    loss     = combined ** NONLINEAR_POWER

    return loss



In [25]:

# ═══════════════════════════════════════════════════════════════
# AW-MAE UTAMA
# ═══════════════════════════════════════════════════════════════
def awmae_score(
    pred_team_goals: np.ndarray,
    pred_opp_goals:  np.ndarray,
    true_team_goals: np.ndarray,
    true_opp_goals:  np.ndarray,
    tournament_names: np.ndarray = None,
) -> dict:
    """
    Hitung AW-MAE.

    Returns dict berisi:
      - awmae        : nilai AW-MAE keseluruhan
      - awmae_detail : DataFrame per baris (untuk analisis)
    """
    n = len(pred_team_goals)
    losses  = np.zeros(n)
    weights = np.ones(n)

    # Bulatkan prediksi ke integer (gol tidak bisa desimal)
    pred_team = np.round(pred_team_goals).astype(int).clip(0)
    pred_opp  = np.round(pred_opp_goals).astype(int).clip(0)
    true_team = np.array(true_team_goals).astype(int)
    true_opp  = np.array(true_opp_goals).astype(int)

    exact_correct   = 0
    outcome_correct = 0
    gd_correct      = 0

    for i in range(n):
        losses[i] = compute_loss(pred_team[i], pred_opp[i], true_team[i], true_opp[i])

        if tournament_names is not None:
            weights[i] = get_tournament_weight(tournament_names[i])

        # Hitung akurasi komponen
        if pred_team[i] == true_team[i] and pred_opp[i] == true_opp[i]:
            exact_correct += 1
        if get_outcome(pred_team[i], pred_opp[i]) == get_outcome(true_team[i], true_opp[i]):
            outcome_correct += 1
        if (pred_team[i] - pred_opp[i]) == (true_team[i] - true_opp[i]):
            gd_correct += 1

    awmae = np.sum(losses * weights) / np.sum(weights)

    detail = {
        "awmae":            round(awmae, 6),
        "exact_accuracy":   round(exact_correct / n * 100, 2),
        "outcome_accuracy": round(outcome_correct / n * 100, 2),
        "gd_accuracy":      round(gd_correct / n * 100, 2),
        "base_mae":         round(
            np.mean([
                (abs(pred_team[i]-true_team[i]) + abs(pred_opp[i]-true_opp[i])) / 2
                for i in range(n)
            ]), 4
        ),
        "n_matches": n,
    }
    return detail


In [26]:


# ═══════════════════════════════════════════════════════════════
# STRATEGI PREDIKSI OPTIMAL UNTUK AW-MAE
# ═══════════════════════════════════════════════════════════════
def predict_awmae_optimal(
    raw_pred_team: np.ndarray,
    raw_pred_opp:  np.ndarray,
    strategy: str = "round",
) -> tuple:
    """
    Konversi prediksi continuous (dari XGB Poisson) ke integer
    dengan berbagai strategi.

    Strategi:
      'round'  : bulatkan biasa (default)
      'floor'  : bulatkan ke bawah (underpredict — bagus jika model overpredict)
      'smart'  : bulatkan biasa, lalu koreksi outcome jika confidence rendah
    """
    pred_team = np.round(raw_pred_team).astype(int).clip(0)
    pred_opp  = np.round(raw_pred_opp).astype(int).clip(0)

    if strategy == "floor":
        pred_team = np.floor(raw_pred_team).astype(int).clip(0)
        pred_opp  = np.floor(raw_pred_opp).astype(int).clip(0)

    elif strategy == "smart":
        # Jika prediksi sangat dekat (beda < 0.3), geser ke seri
        # karena penalty outcome lebih besar dari penalty MAE kecil
        diff = raw_pred_team - raw_pred_opp
        close_match = np.abs(diff) < 0.3

        # Untuk pertandingan hampir seri, rata-ratakan
        avg = (raw_pred_team + raw_pred_opp) / 2
        pred_team = np.where(close_match,
                             np.round(avg).astype(int),
                             np.round(raw_pred_team).astype(int)).clip(0)
        pred_opp  = np.where(close_match,
                             np.round(avg).astype(int),
                             np.round(raw_pred_opp).astype(int)).clip(0)

    return pred_team.astype(int), pred_opp.astype(int)


In [27]:


# ═══════════════════════════════════════════════════════════════
# SIMULASI: BERAPA KONTRIBUSI TIAP KOMPONEN KE AW-MAE
# ═══════════════════════════════════════════════════════════════
def penalty_impact_analysis():
    """
    Hitung berapa AW-MAE untuk skenario prediksi berbeda.
    Berguna untuk memahami mana yang paling worth dioptimasi.
    """
    scenarios = [
        # (pred_team, pred_opp, true_team, true_opp, label)
        (1, 1, 1, 1, "Sempurna"),
        (2, 1, 1, 1, "Meleset 1 gol, outcome benar"),
        (1, 2, 1, 1, "Meleset 1 gol, outcome salah (D→L)"),
        (2, 0, 1, 1, "Meleset 1 gol, GD salah, outcome salah"),
        (0, 0, 1, 1, "Meleset 1 gol, skor beda, outcome benar (D)"),
        (2, 2, 1, 1, "Meleset masing-masing 1, outcome benar (D)"),
        (3, 1, 1, 1, "Meleset 2 gol, outcome benar"),
        (0, 1, 1, 1, "Meleset 1 gol, outcome salah (W→L)"),
    ]

    print(f"\n{'Skenario':<45} {'Loss':>8}")
    print("-" * 55)
    for pt, po, tt, to, label in scenarios:
        loss = compute_loss(pt, po, tt, to)
        print(f"{label:<45} {loss:>8.4f}")


In [28]:


# ═══════════════════════════════════════════════════════════════
# CONTOH PENGGUNAAN
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    print("=" * 55)
    print("ANALISIS DAMPAK TIAP KOMPONEN AW-MAE")
    print("=" * 55)
    penalty_impact_analysis()

    print("\n" + "=" * 55)
    print("CONTOH EVALUASI MODEL")
    print("=" * 55)

    # Simulasi prediksi
    np.random.seed(42)
    n = 1000
    true_team = np.random.poisson(1.5, n)
    true_opp  = np.random.poisson(1.5, n)

    # Model: tebak selalu 1-1
    pred_team_naive = np.ones(n, dtype=int)
    pred_opp_naive  = np.ones(n, dtype=int)

    # Model: prediksi Poisson + pembulatan
    raw_pred_team = np.random.poisson(1.5, n).astype(float) + np.random.normal(0, 0.3, n)
    raw_pred_opp  = np.random.poisson(1.5, n).astype(float) + np.random.normal(0, 0.3, n)
    pred_team_model, pred_opp_model = predict_awmae_optimal(raw_pred_team, raw_pred_opp, "round")

    tournaments = np.random.choice(
        ["FIFA World Cup", "Friendly", "UEFA Euro", "AFC Championship"],
        n
    )

    print("\nBaseline (selalu 1-1):")
    r = awmae_score(pred_team_naive, pred_opp_naive, true_team, true_opp, tournaments)
    for k, v in r.items():
        print(f"  {k:<22}: {v}")

    print("\nModel Poisson + round:")
    r = awmae_score(pred_team_model, pred_opp_model, true_team, true_opp, tournaments)
    for k, v in r.items():
        print(f"  {k:<22}: {v}")

    print("\n" + "=" * 55)
    print("TEMPLATE EVALUASI UNTUK MODEL KAMU")
    print("=" * 55)
    print("""
# Paste ini setelah training XGBoost Poisson:

from awmae_metric import awmae_score, predict_awmae_optimal

# raw_pred adalah output langsung dari model (float)
raw_team = model_team.predict(X_val)
raw_opp  = model_opp.predict(X_val)

# Coba 3 strategi pembulatan
for strategy in ["round", "floor", "smart"]:
    pred_t, pred_o = predict_awmae_optimal(raw_team, raw_opp, strategy)
    result = awmae_score(
        pred_t, pred_o,
        y_val["team_goals"].values,
        y_val["opp_goals"].values,
        tournament_names=X_val["tournament_name"].values  # jika ada
    )
    print(f"Strategy={strategy}: AW-MAE={result['awmae']:.4f}  "
          f"Outcome acc={result['outcome_accuracy']}%  "
          f"Exact acc={result['exact_accuracy']}%")
""")

ANALISIS DAMPAK TIAP KOMPONEN AW-MAE

Skenario                                          Loss
-------------------------------------------------------
Sempurna                                        0.0000
Meleset 1 gol, outcome benar                    2.4150
Meleset 1 gol, outcome salah (D→L)              2.4150
Meleset 1 gol, GD salah, outcome salah          4.0720
Meleset 1 gol, skor beda, outcome benar (D)     1.4822
Meleset masing-masing 1, outcome benar (D)      1.4822
Meleset 2 gol, outcome benar                    4.0720
Meleset 1 gol, outcome salah (W→L)              2.4150

CONTOH EVALUASI MODEL

Baseline (selalu 1-1):
  awmae                 : 3.679164
  exact_accuracy        : 9.5
  outcome_accuracy      : 21.4
  gd_accuracy           : 21.4
  base_mae              : 0.958
  n_matches             : 1000

Model Poisson + round:
  awmae                 : 4.719389
  exact_accuracy        : 5.6
  outcome_accuracy      : 33.1
  gd_accuracy           : 15.6
  base_mae             

In [29]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
import numpy as np

In [30]:
df_train = pd.read_csv('/content/train_clean.csv')
df_test = pd.read_csv('/content/test_clean.csv')

In [31]:
train = df_train.copy()
test = df_test.copy()

test_id = test['Id']

train = train.drop(columns=['id'], errors='ignore')
test = test.drop(columns=['id'], errors='ignore')

In [32]:
target_cols = ['team_goals', 'opp_goals']

X = train.drop(columns=target_cols)
y = train[target_cols]

In [33]:
cat_cols = X.select_dtypes(include=['object', 'category']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

In [35]:
base_model = XGBRegressor(
    objective='count:poisson',   # 🔥 penting!
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42
)

model = MultiOutputRegressor(base_model)

In [36]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', model)
])

In [37]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  Index(['gender', 'is_home', 'neutral', 'tournament', 'population_team',
       'population_opp', 'gdp_per_capita_team', 'gdp_per_capita_opp',
       'altitude_venue', 'distance_travel_team', 'distance_travel_opp',
       'temperature_venue', 'year', 'month', 'match_era',
       'population_team_missing', 'p...
                                                             feature_weights=None,
                                                             gamma=None,
                                                             grow_policy=None,
                                                             importance_type=None,
                                                             interaction_constraints=None,
                                                             learning_rate=0.05,
                                                             max_bin=None,
                                                             max_cat_threshold=None,
                                                             max_cat_to_onehot=None,
                                                             max_delta_step=None,
                                                             max_depth=6,
                                                             max_leaves=None,
                                                             min_child_weight=None,
                                                             missing=nan,
                                                             monotone_constraints=None,
                                                             multi_strategy=None,
                                                             n_estimators=500,
                                                             n_jobs=None,
                                                             num_parallel_tree=None, ...)))])

In [39]:
raw_preds = pipeline.predict(X_val)

raw_team = raw_preds[:, 0]
raw_opp  = raw_preds[:, 1]

In [40]:
results = []

for strategy in ["round", "floor", "smart"]:
    pred_t, pred_o = predict_awmae_optimal(raw_team, raw_opp, strategy)

    res = awmae_score(
        pred_t,
        pred_o,
        y_val["team_goals"].values,
        y_val["opp_goals"].values,
        tournament_names=X_val["tournament_name"].values if "tournament_name" in X_val.columns else None
    )

    print(f"\nStrategy: {strategy}")
    for k, v in res.items():
        print(f"{k}: {v}")

    results.append((strategy, res["awmae"]))


Strategy: round
awmae: 3.301307
exact_accuracy: 8.96
outcome_accuracy: 50.39
gd_accuracy: 23.38
base_mae: 1.0626
n_matches: 15755

Strategy: floor
awmae: 3.31789
exact_accuracy: 11.47
outcome_accuracy: 50.47
gd_accuracy: 22.48
base_mae: 1.0575
n_matches: 15755

Strategy: smart
awmae: 3.308997
exact_accuracy: 9.01
outcome_accuracy: 50.02
gd_accuracy: 23.37
base_mae: 1.0618
n_matches: 15755


In [41]:
best_strategy = sorted(results, key=lambda x: x[1])[0][0]
print("Best strategy:", best_strategy)

Best strategy: round


In [43]:
raw_test = pipeline.predict(test)

final_team, final_opp = predict_awmae_optimal(
    raw_test[:, 0],
    raw_test[:, 1],
    strategy=best_strategy
)

submission = pd.DataFrame({
    "Id": test_id,
    "team_goals": final_team,
    "opp_goals": final_opp
})

In [44]:
submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42422 entries, 0 to 42421
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Id          42422 non-null  object
 1   team_goals  42422 non-null  int64 
 2   opp_goals   42422 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 994.4+ KB


In [46]:
submission.to_csv('submission_gmc_XGBPoisson_AW-MAE.csv',index = False)